In [1]:
# Q8 - Call duration

# Question: how do call lengths differ between customers who subscribed and those who did not?

# `call_duration_seconds` is the length of the last contact. It is not known before a call is made and the outcome is known once
# the call ends (decision 7), so this describes what happened and is not a way to choose whom to call. For customers with several
# calls, only the last call is measured.

In [2]:
import pandas as pd

CLEAN_PATH = "../../data/bank_marketing_clean.parquet"
clean_df = pd.read_parquet(CLEAN_PATH)

clean_df.shape

(41176, 22)

In [3]:
# Q8: length of the last call, by outcome and by call length
subscribed = clean_df["subscribed_term_deposit"] == "yes"
seconds = clean_df["call_duration_seconds"]

print(f"Customers: {len(clean_df)} (should be 41176), subscribers: {subscribed.sum()} (should be 4639)")
print(f"Seconds: min {seconds.min()}, median {seconds.median()}, max {seconds.max()} (should be 0, 180.0, 4918)")
print(f"Calls of 0 seconds: {(seconds == 0).sum()}, of which subscribed: {subscribed[seconds == 0].sum()} (should be 4 and 0)")
print()

# 1. Call length by outcome
by_outcome = (pd.DataFrame({"outcome": clean_df["subscribed_term_deposit"], "seconds": seconds})
              .groupby("outcome")["seconds"]
              .agg(customers="count", median_seconds="median", mean_seconds="mean",
                   lower_quarter_seconds=lambda s: s.quantile(0.25),
                   upper_quarter_seconds=lambda s: s.quantile(0.75),
                   total_hours=lambda s: s.sum() / 3600))
by_outcome["share_of_talk_time_pct"] = (by_outcome["total_hours"] / (seconds.sum() / 3600) * 100).round(1)
by_outcome = by_outcome.round(1)
print("Length of the last call, by outcome:")
print(by_outcome)
print()

# 2. Yes rate by call length
edges = [0, 60, 120, 180, 300, 600, 100000]
labels = ["under 1 min", "1-2 min", "2-3 min", "3-5 min", "5-10 min", "10 min and over"]
length_band = pd.cut(seconds, bins=edges, labels=labels, right=False)

by_length = (pd.DataFrame({"band": length_band, "subscribed": subscribed, "seconds": seconds})
             .groupby("band", observed=True)
             .agg(customers=("subscribed", "count"), subscribers=("subscribed", "sum"),
                  total_hours=("seconds", lambda s: s.sum() / 3600)))
by_length["yes_rate_pct"] = (by_length["subscribers"] / by_length["customers"] * 100).round(2)
by_length["share_of_customers_pct"] = (by_length["customers"] / len(clean_df) * 100).round(1)
by_length["share_of_subscribers_pct"] = (by_length["subscribers"] / subscribed.sum() * 100).round(1)
by_length["share_of_talk_time_pct"] = (by_length["total_hours"] / (seconds.sum() / 3600) * 100).round(1)
by_length["total_hours"] = by_length["total_hours"].round(1)
by_length["small_group"] = by_length["customers"] < 100

print(f"Customers in bands: {by_length['customers'].sum()} (should be 41176), "
      f"subscribers: {by_length['subscribers'].sum()} (should be 4639)")
print("Yes rate by length of the last call:")
by_length

Customers: 41176 (should be 41176), subscribers: 4639 (should be 4639)
Seconds: min 0, median 180.0, max 4918 (should be 0, 180.0, 4918)
Calls of 0 seconds: 4, of which subscribed: 0 (should be 4 and 0)

Length of the last call, by outcome:
         customers  median_seconds  mean_seconds  lower_quarter_seconds  \
outcome                                                                   
no           36537           164.0         220.9                   95.0   
yes           4639           449.0         553.3                  253.5   

         upper_quarter_seconds  total_hours  share_of_talk_time_pct  
outcome                                                              
no                       279.0       2241.6                    75.9  
yes                      741.5        712.9                    24.1  

Customers in bands: 41176 (should be 41176), subscribers: 4639 (should be 4639)
Yes rate by length of the last call:


,customers,subscribers,total_hours,yes_rate_pct,share_of_customers_pct,share_of_subscribers_pct,share_of_talk_time_pct,small_group
band,,,,,,,,
under 1 min,4179,1,40.3,0.02,10.1,0.0,1.4,False
1-2 min,8612,163,215.4,1.89,20.9,3.5,7.3,False
2-3 min,7788,394,320.7,5.06,18.9,8.5,10.9,False
3-5 min,9349,950,602.6,10.16,22.7,20.5,20.4,False
5-10 min,7770,1441,889.7,18.55,18.9,31.1,30.1,False
10 min and over,3478,1690,885.8,48.59,8.4,36.4,30.0,False
